# Hot Oceans Album: Warm Water and Tropical Cyclones

Issue #21. Follows [notebook-guide.md](../docs/notebook-guide.md) and [data-guide.md](../docs/data-guide.md). Every source is listed in [data/hot_oceans/sources.md](../data/hot_oceans/sources.md).

EIC page [5097 2022 Hurricane Season](https://svs.gsfc.nasa.gov/5097/) draws a season of storm tracks over sea surface temperature and says that warm water makes and feeds hurricanes. This notebook checks that message in the data behind the page, for the four ocean basins the album plays (Atlantic, West Pacific, South Indian, South Pacific), and looks for what in it can become music:

1. **Where are storms born?** The sea surface temperature under each storm when it forms, against the 26.5 °C line EIC draws.
2. **Does warmer water make stronger storms?** How a storm's wind changes over the next 24 hours, against the water under it, and how each storm's energy relates to the water it crossed.
3. **What do high-energy storms look like?** Peak wind, lifetime, distance travelled, speed and rapid intensification of the most energetic storms against the rest.
4. **Are hot seasons stormier?** Each season's water temperature in the basin's main development region against the season's storm count and energy, and where 2023 sits.

5. **What can become music?** The findings above as sound, combined with the two earlier designs: the season of storms as voices (latitude as pitch, east–west travel as panning) and the single storm as rhythm (wind speed as tempo).

**Years.** The storm-by-storm analysis (sections 4–7) uses eight years, 2003–2006 and 2021–2024: the first and last four of the MUR record that JTWC has published, about 500 named storms. The season comparison (section 8) uses every season from 2003 to 2024, but only the peak months of each basin. Each monthly map takes ERDDAP 30–60 seconds to cut, so this keeps the first run near an hour instead of three.

Run with `npm run notebook`. Downloads go to `data/hot_oceans/raw/` (local only). No login is needed. The first run downloads 194 monthly sea surface temperature maps (about 450 MB); reruns read the local files.

In [ ]:
import html
import io
import json
import re
import zipfile
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from IPython.display import Markdown, display
from scipy.io import netcdf_file
from scipy.stats import spearmanr

pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"figure.dpi": 72, "axes.spines.top": False, "axes.spines.right": False})

RAW = Path("../data/hot_oceans/raw")
for sub in ("api", "jtwc", "mur_monthly"):
    (RAW / sub).mkdir(parents=True, exist_ok=True)

SVS_ID = 5097       # 2022 Hurricane Season: storm tracks over sea surface temperature
WIND_SVS_ID = 5072  # Atlantic Hurricane Wind Speed Plots: the season as wind speed over time
EIC_GALLERY = 40467
YEARS = (2003, 2024)  # MUR starts in June 2002; JTWC has not published 2025 yet
STORM_YEARS = (2003, 2004, 2005, 2006, 2021, 2022, 2023, 2024)  # storm-by-storm analysis (sections 4-7)
PEAK_MONTHS = (1, 2, 3, 7, 8, 9, 10)  # the basins' peak months, for the season comparison (section 8)
WARM_C = 26.5         # the threshold EIC names on 5097

HURDAT2_URL = "https://www.nhc.noaa.gov/data/hurdat/hurdat2-1851-2025-092326.txt"
JTWC_URL = "https://www.metoc.navy.mil/jtwc/products/best-tracks/{y}/{y}s-b{b}/b{b}{y}.zip"
ERDDAP = "https://coastwatch.pfeg.noaa.gov/erddap/griddap"
MUR_DATASET = "jplMURSST41mday"
MUR_URL = (f"{ERDDAP}/{MUR_DATASET}.nc?sst%5B({{y}}-{{m:02d}}-16T00:00:00Z)%5D"
           "%5B(-50):25:(55)%5D%5B(-179.99):25:(180)%5D")


def download(name, url, timeout=600):
    """Download url into data/hot_oceans/raw once and return the local path."""
    path = RAW / name
    if not path.exists():
        response = requests.get(url, timeout=timeout)
        response.raise_for_status()
        tmp = path.with_suffix(path.suffix + ".part")
        tmp.write_bytes(response.content)
        tmp.rename(path)
    return path


def svs_page(page_id):
    return json.loads(download(f"api/{page_id}.json", f"https://svs.gsfc.nasa.gov/api/{page_id}").read_text())


def plain(text):
    return re.sub(r"\s+", " ", html.unescape(re.sub(r"<[^>]+>", " ", text or ""))).strip()


def source_note(fig, text):
    fig.text(0.01, -0.02, text, fontsize=8, color="gray", ha="left", va="top")


TRACK_CREDIT = "Tracks: NOAA NHC HURDAT2 (Atlantic), JTWC best tracks (other basins)."
SST_CREDIT = "SST: GHRSST MUR v4.1 monthly (NASA JPL), via NOAA CoastWatch ERDDAP."
CREDIT = TRACK_CREDIT + " " + SST_CREDIT
BASINS = ["Atlantic", "West Pacific", "South Indian", "South Pacific"]
BASIN_COLOR = {"Atlantic": "#e8785a", "West Pacific": "#4c8bd6", "South Indian": "#9a6fd0", "South Pacific": "#3aa58a"}

## 1. Links

Everything this notebook reads, in one place. The tables are built from the SVS API, so they stay correct if EIC changes.

### 1.1 Where the visualizations sit in the EIC gallery

In [ ]:
def gallery_paths(gallery_id):
    """Map each page id to the gallery sections that lead to it from the EIC gallery."""
    paths = {}

    def walk(page_id, trail):
        page = svs_page(page_id)
        if page.get("page_type") != "Gallery":
            paths.setdefault(page_id, []).append(trail)
            return
        for group in page.get("media_groups", []):
            for item in group.get("items", []):
                inst = item.get("instance") or {}
                if item["type"] in ("gallery_page", "details_page") and inst.get("id"):
                    step = f"{group['title']} → [{inst.get('title')}]({inst['url'].replace('http://', 'https://')})"
                    if inst.get("page_type") == "Gallery" and inst["id"] not in [t[0] for t in trail]:
                        walk(inst["id"], trail + [(inst["id"], step)])
                    elif inst.get("page_type") != "Gallery":
                        paths.setdefault(inst["id"], []).append(trail + [(inst["id"], step)])

    walk(gallery_id, [])
    return paths


paths = gallery_paths(EIC_GALLERY)
rows = ["| SVS page | Path from the [EIC gallery](https://svs.gsfc.nasa.gov/gallery/earth-information-center/) |", "| --- | --- |"]
for page_id in (SVS_ID, WIND_SVS_ID):
    trails = {" › ".join(step for _, step in trail) for trail in paths.get(page_id, [])}
    rows.append(f"| [{page_id}](https://svs.gsfc.nasa.gov/{page_id}/) | {'<br>'.join(sorted(trails))} |")
display(Markdown("\n".join(rows)))

### 1.2 What the pages cite, and what this notebook downloads

- **Cited by EIC** comes from the `datasets` field of each SVS page, as published.
- **Downloaded here** is what this notebook actually reads.

In [ ]:
JTWC_PAGE = "https://www.metoc.navy.mil/jtwc/jtwc.html?best-tracks"
DOWNLOADED = {
    "HURDAT2": (f"[HURDAT2]({HURDAT2_URL}) (NOAA NHC, revised 2026-09-23)",
                "The dataset 5072 cites. NHC's post-season best tracks, which replace the ATCF best tracks once a season is reviewed. Atlantic only. Public."),
    "Automated Tropical Cyclone Forecast (ATCF) - Best Track": (
        f"[JTWC best tracks]({JTWC_PAGE}), ATCF b-deck format: `{JTWC_URL.format(y='<year>', b='<wp|sh>')}`",
        "Substitute for the other basins. The NHC ATCF archive that EIC links covers only the Atlantic and the eastern and central Pacific; "
        "the US Joint Typhoon Warning Center keeps the ATCF best tracks of the West Pacific and the Southern Hemisphere. Public."),
    "Multi-scale Ultra-high Resolution (MUR) Sea Surface Temperature (SST) Analysis": (
        f"[MUR SST v4.1 monthly composite]({ERDDAP}/{MUR_DATASET}.html) on NOAA CoastWatch ERDDAP, every 25th grid point (0.25°), 50° S–55° N",
        "The dataset 5097 cites, served without a login by NOAA CoastWatch. ERDDAP averages the daily MUR v4.1 analysis into months. "
        "PO.DAAC's own copy needs an Earthdata login and is about 400 MB a day. Public."),
}
rows = ["| SVS page | Cited by EIC | Downloaded here | Note |", "| --- | --- | --- | --- |"]
for page_id in (SVS_ID, WIND_SVS_ID):
    page = svs_page(page_id)
    for d in page["datasets"]:
        name = d["name"] or d["common_name"]
        cited = f"[{name}]({d['url'].replace('http://', 'https://')})" if d.get("url") else f"{name}, no link"
        used, note = DOWNLOADED.get(name, ("nothing", "Not about storm position, strength or water temperature. Not analyzed."))
        rows.append(f"| [{page_id}](https://svs.gsfc.nasa.gov/{page_id}/) | {cited} | {used} | {note} |")
display(Markdown("\n".join(rows)))

## 2. What EIC says

The captions of SVS 5097, quoted as published, and the description of the EIC gallery it sits in ([NASA and Hurricanes](https://svs.gsfc.nasa.gov/gallery/nasaand-hurricanes/), 40487).

In [ ]:
captions = [plain(g.get("caption")) or plain(g.get("description")) for g in svs_page(SVS_ID)["media_groups"]]
gallery = plain(svs_page(40487)["description"])
display(Markdown("\n\n".join(f"> {c}" for c in captions if c) + f"\n\n**Gallery 40487:**\n\n> {gallery}"))

**In short (our reading):**

- **The message**: warm water makes and feeds hurricanes. Two claims, in EIC's words: "Hurricanes tend to form over the warm tropical oceans where the SST is warmer than ~26.5 C (~80 F)", and "Sea surface temperatures play an important role in hurricane formation and development, with warmer temperatures linked to more intense storms." The page belongs to the SVS series *Hurricane-SST Connection*.
- **The picture**: one season of storm tracks drawn over the sea surface temperature, with the storm's category along each track.
- **The wider claim** (gallery 40487): "Hurricanes are getting stronger due to climate change warming the air and oceans."
- **What it does not say**: nothing about which way storms move or where they make landfall. The tracks show where storms went and how strong they were over which water.

This notebook checks the two claims of 5097, then looks at what the strongest storms look like and whether hot seasons are stormier. The last two are ours.

## 3. Source data

### 3.1 Storm tracks

[HURDAT2](https://www.nhc.noaa.gov/data/#hurdat) · [JTWC best tracks](https://www.metoc.navy.mil/jtwc/jtwc.html?best-tracks)

- **What it is**: the best estimate, made after each season, of every tropical cyclone's centre position, maximum sustained wind and minimum pressure, every 6 hours.
- **Provider**: NOAA National Hurricane Center (Atlantic), US Joint Typhoon Warning Center (West Pacific, South Indian, South Pacific). Both are US government works in the public domain.
- **Units**: degrees, knots (1-minute sustained wind, 1 kt = 1.852 km/h), hPa, UTC.

What we keep:

- **Basins**: the JTWC Southern Hemisphere file is split at 135° E by where each storm forms: west of it is the South Indian Ocean, east of it the South Pacific, as IBTrACS does.
- **Stages**: only the tropical and subtropical stages (`TD TS HU TY ST TC SD SS`). Extratropical, low and disturbance stages are dropped: they are no longer driven by warm water.
- **Times**: only the synoptic hours (00, 06, 12, 18 UTC), so that extra landfall fixes in HURDAT2 do not weigh more.
- **Storms**: those that reach 34 kt (named storms) and lie wholly within 2003–2024 (`rec_all`, for the season comparison). The storm-by-storm analysis keeps those whose every month has an SST map (`rec`, section 3.2).

In [ ]:
TROPICAL = {"TD", "TS", "HU", "TY", "ST", "TC", "SD", "SS"}


def read_hurdat2(path):
    rows, sid, name = [], None, None
    for line in Path(path).read_text().splitlines():
        p = [s.strip() for s in line.split(",")]
        if p[0][:2].isalpha() and len(p) <= 4:  # header line of a storm
            sid, name = p[0], p[1]
            continue
        lat = float(p[4][:-1]) * (1 if p[4][-1] == "N" else -1)
        lon = float(p[5][:-1]) * (1 if p[5][-1] == "E" else -1)
        rows.append((sid, name, pd.Timestamp(f"{p[0]}T{p[1][:2]}:{p[1][2:]}"), p[3], lat, lon, int(p[6]), int(p[7]), "AL"))
    return pd.DataFrame(rows, columns=["sid", "name", "t", "stage", "lat", "lon", "vmax_kt", "mslp_hpa", "bb"])


def read_jtwc(path):
    """One ATCF b-deck zip of JTWC: one file per storm, one line per time and wind radius."""
    rows = []
    with zipfile.ZipFile(path) as z:
        for n in z.namelist():
            for line in z.read(n).decode("latin-1").splitlines():
                p = [s.strip() for s in line.split(",")]
                if len(p) < 11 or not p[2].isdigit():
                    continue
                try:
                    lat = float(p[6][:-1]) / 10 * (1 if p[6][-1] == "N" else -1)
                    lon = float(p[7][:-1]) / 10 * (1 if p[7][-1] == "E" else -1)
                    vmax = int(p[8])
                except ValueError:
                    continue
                mslp = int(p[9]) if p[9].isdigit() else np.nan
                rows.append((f"{p[0]}{p[1]}{Path(n).stem[-4:]}", "", pd.Timestamp(f"{p[2][:8]}T{p[2][8:10]}"), p[10],
                             lat, lon, vmax, mslp, p[0]))
    df = pd.DataFrame(rows, columns=["sid", "name", "t", "stage", "lat", "lon", "vmax_kt", "mslp_hpa", "bb"])
    return df.drop_duplicates(["sid", "t"])


hurdat = read_hurdat2(download(Path(HURDAT2_URL).name, HURDAT2_URL))
jtwc = pd.concat([read_jtwc(download(f"jtwc/b{b}{y}.zip", JTWC_URL.format(y=y, b=b)))
                  for y in range(YEARS[0], YEARS[1] + 1) for b in ("wp", "sh")])

first = jtwc.sort_values("t").groupby("sid").first()
basin_of = pd.Series(np.where(first["bb"] == "SH", np.where(first["lon"] % 360 < 135, "South Indian", "South Pacific"),
                              "West Pacific"), index=first.index)
jtwc["basin"] = jtwc["sid"].map(basin_of)
hurdat["basin"] = "Atlantic"

rec = pd.concat([hurdat, jtwc], ignore_index=True)
rec = rec[rec["t"].dt.hour.isin([0, 6, 12, 18]) & (rec["t"].dt.minute == 0) & rec["stage"].isin(TROPICAL)]
span = rec.groupby("sid")["t"].agg(["min", "max"])
inside = span.index[(span["min"].dt.year >= YEARS[0]) & (span["max"].dt.year <= YEARS[1])]
named = rec.groupby("sid")["vmax_kt"].max().loc[lambda v: v >= 34].index
rec_all = rec[rec["sid"].isin(inside.intersection(named))].sort_values(["sid", "t"]).reset_index(drop=True)
rec_all["name"] = rec_all["sid"].map(hurdat.groupby("sid")["name"].first()).fillna("")

MONTHS = pd.PeriodIndex(sorted({pd.Period(f"{y}-{m:02d}", "M") for y in range(YEARS[0], YEARS[1] + 1) for m in range(1, 13)
                                if y in STORM_YEARS or m in PEAK_MONTHS}))
covered = rec_all.groupby("sid")["t"].agg(lambda t: t.dt.to_period("M").isin(MONTHS).all() and t.dt.year.isin(STORM_YEARS).any())
rec = rec_all[rec_all["sid"].map(covered)].reset_index(drop=True)
pd.DataFrame({"storms 2003–2024": rec_all.groupby("basin")["sid"].nunique(),
              "storms analysed one by one": rec.groupby("basin")["sid"].nunique(),
              "their 6-hourly records": rec.groupby("basin").size()}).loc[BASINS]

### 3.2 Sea surface temperature

[MUR SST v4.1](https://podaac.jpl.nasa.gov/dataset/MUR-JPL-L4-GLOB-v4.1) · [ERDDAP jplMURSST41mday](https://coastwatch.pfeg.noaa.gov/erddap/griddap/jplMURSST41mday.html)

- **What it is**: the Multi-scale Ultra-high Resolution analysis of the foundation sea surface temperature (the water just below the daily surface warming), from several satellites, on a 0.01° grid. 5097 draws it under the storms.
- **Provider**: NASA JPL (GHRSST), served as monthly means by NOAA CoastWatch ERDDAP.
- **What we download**: one map per month (every month of the eight storm years, the peak months of the others), every 25th grid point (0.25°), 50° S to 55° N. Land and sea ice are empty.
- **Units**: °C.

A monthly mean is enough to tell warm water from cool water under a storm, and smooths over the cold wake a storm leaves behind it. Section 5.1 checks that the result holds with the month before as well.

In [ ]:
LAT0, LON0, STEP = -50.0, -179.99, 0.25
GRID_LAT = LAT0 + STEP * np.arange(421)
GRID_LON = LON0 + STEP * np.arange(1440)


def mur_path(month):
    return download(f"mur_monthly/mur_mday_{month.year}{month.month:02d}.nc", MUR_URL.format(y=month.year, m=month.month))


def read_mur(month):
    with netcdf_file(mur_path(month), "r", mmap=False) as f:
        sst = f.variables["sst"][0].astype("float32")
    sst[sst < -100] = np.nan
    return sst


for month in MONTHS:  # first run: about an hour
    mur_path(month)
print(f"{len(MONTHS)} monthly maps, {sum(mur_path(m).stat().st_size for m in MONTHS) / 1e6:.0f} MB")

### 3.3 The water under each storm

Each 6-hourly record gets the SST of the nearest grid cell in its month (`sst_c`) and in the month before (`sst_prev_c`). Records over land get none. Then, for every record, what the storm does next:

- `dv24_kt`: the change of maximum wind over the next 24 hours
- `speed_kmh`, `west_km`, `pole_km`: how fast and which way it moves over the next 24 hours (towards the west, and towards the pole of its own hemisphere)

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))


def sample_sst(sst, lat, lon):
    i = np.clip(np.rint((np.asarray(lat) - LAT0) / STEP).astype(int), 0, len(GRID_LAT) - 1)
    j = np.rint(((np.asarray(lon) - LON0) % 360) / STEP).astype(int) % len(GRID_LON)
    return sst[i, j]


rec["sst_c"] = np.nan
rec["sst_prev_c"] = np.nan
month_of = rec["t"].dt.to_period("M")
for month in MONTHS:
    sst = read_mur(month)
    for col, m in (("sst_c", month_of), ("sst_prev_c", month_of - 1)):
        idx = rec.index[m == month]
        rec.loc[idx, col] = sample_sst(sst, rec.loc[idx, "lat"], rec.loc[idx, "lon"])

g = rec.groupby("sid")
ahead = g[["t", "lat", "lon", "vmax_kt", "sst_c"]].shift(-4)
whole_day = (ahead["t"] - rec["t"]) == pd.Timedelta("24h")
rec["dv24_kt"] = (ahead["vmax_kt"] - rec["vmax_kt"]).where(whole_day)
rec["sst_next24_c"] = ahead["sst_c"].where(whole_day)
rec["speed_kmh"] = (haversine_km(rec["lat"], rec["lon"], ahead["lat"], ahead["lon"]) / 24).where(whole_day)
hemi = np.sign(rec["lat"]).replace(0, 1)
rec["pole_km"] = ((ahead["lat"] - rec["lat"]) * hemi * 111.2).where(whole_day)
rec["west_km"] = (-((ahead["lon"] - rec["lon"] + 180) % 360 - 180) * 111.2 * np.cos(np.radians(rec["lat"]))).where(whole_day)
rec["warm"] = rec["sst_c"] >= WARM_C
print(f"records over water: {rec['sst_c'].notna().mean():.1%}")

### 3.4 Looking like 5097

The 2023 Atlantic season on the September 2023 SST, as 5097 draws 2022: the tracks over warm and cool water, the 26.5 °C line in white.

In [ ]:
def sst_map(ax, month, lon_range, lat_range):
    sst = read_mur(pd.Period(month, "M"))
    lon360 = GRID_LON % 360
    order = np.argsort(lon360)
    im = ax.pcolormesh(lon360[order], GRID_LAT, sst[:, order], cmap="RdYlBu_r", vmin=18, vmax=31, shading="auto")
    ax.contour(lon360[order], GRID_LAT, np.nan_to_num(sst[:, order], nan=-5), levels=[WARM_C], colors="white", linewidths=1.2)
    ax.set(xlim=lon_range, ylim=lat_range, aspect="equal", xlabel="longitude (°E)", ylabel="latitude (°)")
    return im


CATEGORY_KT = [34, 64, 83, 96, 113, 137]  # tropical storm, categories 1-5 (Saffir-Simpson)
fig, ax = plt.subplots(figsize=(12, 6), layout="constrained")
im = sst_map(ax, "2023-09", (255, 350), (5, 55))
for sid, d in rec[(rec["basin"] == "Atlantic") & (rec["t"].dt.year == 2023)].groupby("sid"):
    ax.plot(d["lon"] % 360, d["lat"], color="black", linewidth=0.8)
    ax.scatter(d["lon"] % 360, d["lat"], c=d["vmax_kt"], cmap="magma_r", vmin=20, vmax=140, s=8, zorder=3)
    if d["vmax_kt"].max() >= 64:
        top = d.loc[d["vmax_kt"].idxmax()]
        ax.annotate(top["name"].title(), (top["lon"] % 360, top["lat"]), fontsize=8, xytext=(3, 3), textcoords="offset points")
fig.colorbar(im, ax=ax, shrink=0.7, label="SST, September 2023 (°C)")
ax.set_title("2023 Atlantic storms (dots darker = stronger) were born south of the 26.5 °C line and faded north of it")
source_note(fig, CREDIT)

## 4. Message 1: storms are born over warm water

> "Hurricanes tend to form over the warm tropical oceans where the SST is warmer than ~26.5 C (~80 F)" (5097)

The SST under the first record of each storm.

In [ ]:
storms = g.agg(basin=("basin", "first"), name=("name", "first"), start=("t", "min"), end=("t", "max"),
               gen_lat=("lat", "first"), gen_lon=("lon", "first"), gen_sst=("sst_c", "first"),
               lmi_kt=("vmax_kt", "max"), min_mslp_hpa=("mslp_hpa", "min"))
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2), sharex=True, sharey=True, layout="constrained")
rows = []
for ax, b in zip(axes, BASINS):
    x = storms.loc[storms["basin"] == b, "gen_sst"].dropna()
    ax.hist(x, bins=np.arange(22, 32, 0.5), color=BASIN_COLOR[b])
    ax.axvline(WARM_C, color="black", linestyle="--", linewidth=1)
    ax.set(title=b, xlabel="SST at birth (°C)")
    rows.append({"basin": b, "storms": len(x), "born at or above 26.5 °C": f"{(x >= WARM_C).mean():.0%}", "median SST (°C)": round(x.median(), 1)})
axes[0].set_ylabel("storms")
fig.suptitle("Almost every storm is born over water warmer than 26.5 °C (dashed)")
source_note(fig, CREDIT)
pd.DataFrame(rows).set_index("basin")

## 5. Message 2: warmer water, stronger storms

> "Sea surface temperatures play an important role in hurricane formation and development, with warmer temperatures linked to more intense storms." (5097)

### 5.1 What the water under a storm says about its next 24 hours

For every record over water, the change of maximum wind over the next 24 hours, grouped by the SST under the storm. Records whose next 24 hours end over land are left out, so landfall does not count as weakening.

In [ ]:
over_water = rec.dropna(subset=["dv24_kt", "sst_c", "sst_next24_c"])
SST_BINS = np.arange(21, 32, 1.0)


def by_sst(d, col="sst_c"):
    """Mean 24-hour wind change and share strengthening per 1 °C of SST, for bins with 30 records or more."""
    b = pd.cut(d[col], SST_BINS)
    out = d.groupby(b, observed=True)["dv24_kt"].agg(records="size", mean_kt="mean", strengthening=lambda v: (v > 0).mean())
    out.index = [i.mid for i in out.index]
    return out[out["records"] >= 30]


def neutral_sst(d):
    """The SST at which the mean 24-hour change crosses zero, by linear interpolation between 0.5 °C bins."""
    b = pd.cut(d["sst_c"], np.arange(22, 31.5, 0.5))
    m = d.groupby(b, observed=True)["dv24_kt"].agg(["mean", "size"])
    m = m[m["size"] >= 30]
    mids = np.array([i.mid for i in m.index])
    up = np.flatnonzero((m["mean"].values[:-1] < 0) & (m["mean"].values[1:] >= 0))
    if not len(up):
        return np.nan
    k = up[-1]
    y0, y1 = m["mean"].values[k], m["mean"].values[k + 1]
    return mids[k] + (mids[k + 1] - mids[k]) * (-y0) / (y1 - y0)


fig, axes = plt.subplots(1, 2, figsize=(14, 4.2), layout="constrained")
for b in BASINS:
    t = by_sst(over_water[over_water["basin"] == b])
    axes[0].plot(t.index, t["mean_kt"], marker="o", color=BASIN_COLOR[b], label=b)
all_t = by_sst(over_water)
axes[0].axhline(0, color="gray", linewidth=0.8)
axes[0].axvline(WARM_C, color="black", linestyle="--", linewidth=0.8)
axes[0].set(title="Over cooler water storms weaken, over warmer water they strengthen", xlabel="SST under the storm (°C, 1 °C bins)",
            ylabel="mean wind change over next 24 h (kt)")
axes[0].legend(fontsize=8)
axes[1].bar(all_t.index, all_t["strengthening"], width=0.8, color="#c46a3c")
axes[1].axhline(0.5, color="gray", linewidth=0.8)
axes[1].axvline(WARM_C, color="black", linestyle="--", linewidth=0.8)
axes[1].set(title="Share of records that strengthen over the next 24 h (all basins)", xlabel="SST under the storm (°C, 1 °C bins)", ylim=(0, 1))
source_note(fig, CREDIT)

checks = pd.DataFrame({
    "all records": {b: neutral_sst(over_water[over_water["basin"] == b]) for b in BASINS},
    "latitude 10–25° only": {b: neutral_sst(over_water[(over_water["basin"] == b) & over_water["lat"].abs().between(10, 25)]) for b in BASINS},
    "SST of the month before": {b: neutral_sst(over_water[over_water["basin"] == b].assign(sst_c=lambda d: d["sst_prev_c"])) for b in BASINS},
}).round(1)
display(Markdown("**The neutral SST**: where the mean 24-hour change turns from weakening to strengthening (°C)"))
checks

### 5.2 When does the wind answer the water?

Does the SST under a storm tell about the wind change to come, or about the change around it now? The correlation between the SST under the storm and its 24-hour wind change, with the 24-hour window shifted from 48 hours before to 72 hours after. A shift of −12 h is the window centred on the moment of the SST.

In [ ]:
def lagged(rec, hours):
    """Spearman correlation between SST at t and the wind change from t + hours to t + hours + 24 h."""
    k = hours // 6
    g = rec.groupby("sid")
    v0, v1 = g["vmax_kt"].shift(-k), g["vmax_kt"].shift(-k - 4)
    t0, t1 = g["t"].shift(-k), g["t"].shift(-k - 4)
    ok = ((t0 - rec["t"]) == pd.Timedelta(hours=hours)) & ((t1 - t0) == pd.Timedelta("24h"))
    d = pd.DataFrame({"sst": rec["sst_c"], "dv": (v1 - v0)})[ok].dropna()
    return spearmanr(d["sst"], d["dv"]).statistic, len(d)


LAGS = list(range(-48, 73, 6))
lag = pd.DataFrame([(b, h, *lagged(rec[rec["basin"] == b], h)) for b in BASINS for h in LAGS], columns=["basin", "hours", "rho", "n"])
fig, ax = plt.subplots(figsize=(10, 3.8), layout="constrained")
for b, d in lag.groupby("basin"):
    ax.plot(d["hours"], d["rho"], marker=".", color=BASIN_COLOR[b], label=b)
ax.axvline(0, color="gray", linewidth=0.8)
ax.set(title="The wind changes with the water under it: the link is closest around the present moment", xlabel="start of the 24-hour wind change, hours after the SST",
       ylabel="Spearman ρ (SST, wind change)")
ax.legend(fontsize=8)
source_note(fig, CREDIT)
lag.loc[lag.groupby("basin")["rho"].idxmax()].set_index("basin").loc[BASINS].round(2)

### 5.3 Fuel: hours over warm water

Per storm. Does the water at birth, the average water under the track, or the time spent over warm water go with how strong and how energetic a storm gets?

- **Energy** is the accumulated cyclone energy (ACE): the sum of the squared maximum wind (kt²) of every 6-hourly record at 35 kt or more, divided by 10⁴. It grows with both strength and lifetime.
- **Warm hours**: hours at tropical storm strength (34 kt) or more over water at or above 26.5 °C. A storm that lives longer has more of them, so the **share** of its named life over warm water is shown as well.

In [ ]:
rec["ace"] = np.where(rec["vmax_kt"] >= 35, rec["vmax_kt"] ** 2 / 1e4, 0.0)
g = rec.groupby("sid")
named_rec = rec[rec["vmax_kt"] >= 34]
step = haversine_km(g["lat"].shift(), g["lon"].shift(), rec["lat"], rec["lon"]).where((rec["t"] - g["t"].shift()) == pd.Timedelta("6h"))
storms = storms.assign(
    ace=g["ace"].sum(),
    life_days=(storms["end"] - storms["start"]).dt.total_seconds() / 86400,
    ts_days=named_rec.groupby("sid").size() * 0.25,
    track_km=step.groupby(rec["sid"]).sum(),
    max_dv24_kt=g["dv24_kt"].max(),
    track_sst=named_rec.groupby("sid")["sst_c"].mean(),
    warm_hours=named_rec[named_rec["warm"]].groupby("sid").size().reindex(storms.index, fill_value=0) * 6,
    abs_gen_lat=storms["gen_lat"].abs(),
)
storms["warm_share"] = storms["warm_hours"] / (storms["ts_days"] * 24)
storms["speed_kmh"] = storms["track_km"] / (storms["life_days"] * 24)
storms["season"] = np.where(storms["basin"].str.startswith("South") & (storms["start"].dt.month >= 7),
                            storms["start"].dt.year + 1, storms["start"].dt.year)

DRIVERS = {"gen_sst": "SST at birth", "track_sst": "mean SST under the track", "warm_hours": "hours over warm water",
           "warm_share": "share of its named life over warm water", "abs_gen_lat": "latitude of birth"}
OUTCOMES = {"lmi_kt": "peak wind", "ace": "energy (ACE)", "ts_days": "days as a named storm", "track_km": "distance travelled",
            "speed_kmh": "mean forward speed", "max_dv24_kt": "fastest 24-h strengthening"}
corr = pd.DataFrame({DRIVERS[x]: {OUTCOMES[y]: spearmanr(storms[x], storms[y], nan_policy="omit").statistic for y in OUTCOMES}
                     for x in DRIVERS}).round(2)
display(Markdown(f"**Spearman ρ, {len(storms)} storms** (rows: what the storm did; columns: what it had)"))
corr

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), layout="constrained")
for b in BASINS:
    d = storms[storms["basin"] == b]
    axes[0].scatter(d["warm_hours"] / 24, d["ace"], s=6, alpha=0.5, color=BASIN_COLOR[b], label=b)
    axes[1].scatter(d["warm_hours"] / 24, d["lmi_kt"], s=6, alpha=0.5, color=BASIN_COLOR[b])
    axes[2].scatter(d["track_sst"], d["lmi_kt"], s=6, alpha=0.5, color=BASIN_COLOR[b])
axes[0].set(yscale="log", xlabel="days over water ≥ 26.5 °C", ylabel="ACE (10⁴ kt²)", title="Energy grows with days over warm water")
axes[1].set(xlabel="days over water ≥ 26.5 °C", ylabel="peak wind (kt)", title="So does peak wind")
axes[2].set(xlabel="mean SST under the track (°C)", ylabel="peak wind (kt)", title="The mean SST alone says little")
axes[0].legend(fontsize=8, markerscale=3)
for ax in axes:
    source_note(fig, CREDIT)

## 6. What high-energy storms look like

Storms ranked by ACE within their own basin: the top 10 % against the bottom half.

In [ ]:
storms["ace_rank"] = storms.groupby("basin")["ace"].rank(pct=True)
storms["group"] = np.select([storms["ace_rank"] >= 0.9, storms["ace_rank"] <= 0.5], ["top 10 %", "bottom 50 %"], "middle")
FEATURES = {"lmi_kt": "peak wind (kt)", "min_mslp_hpa": "lowest pressure (hPa)", "ts_days": "days as a named storm",
            "track_km": "distance travelled (km)", "speed_kmh": "mean forward speed (km/h)", "max_dv24_kt": "fastest 24-h strengthening (kt)",
            "warm_hours": "hours over water ≥ 26.5 °C", "gen_sst": "SST at birth (°C)", "abs_gen_lat": "latitude of birth (°)"}
table = storms.groupby("group")[list(FEATURES)].median().T.rename(index=FEATURES)[["bottom 50 %", "middle", "top 10 %"]]
table.loc["rapid intensification (≥ 30 kt in 24 h)"] = storms.groupby("group")["max_dv24_kt"].agg(lambda v: (v >= 30).mean())[table.columns]
table.loc["storms"] = storms["group"].value_counts()[table.columns]
display(Markdown("**Medians** (rapid intensification: share of storms)"))
table.round(2)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6.5), layout="constrained")
for ax, col in zip(axes.flat, ["lmi_kt", "ts_days", "track_km", "speed_kmh", "max_dv24_kt", "warm_hours", "gen_sst", "abs_gen_lat"]):
    data = [storms.loc[storms["group"] == k, col].dropna() for k in ("bottom 50 %", "middle", "top 10 %")]
    ax.boxplot(data, showfliers=False, widths=0.6)
    ax.set_xticks([1, 2, 3], ["bottom\n50 %", "middle", "top\n10 %"])
    ax.set_title(FEATURES[col], fontsize=10)
fig.suptitle("High-energy storms live longer, travel farther, spin up faster and stay longer over warm water; they move no faster")
source_note(fig, CREDIT)

## 7. The life of a storm: wind, water, direction and speed together

5097 shows each storm as a track over the water. Here every storm that leaves warm water over the sea is lined up at the moment it does: the first record under 26.5 °C after its last record at or above it. Around that moment: its wind, the water under it, its latitude, its forward speed and which way it moves. Medians over the storms, with the middle half shaded.

In [ ]:
def exit_time(d):
    """Time of the first record below 26.5 °C after the storm's last record over warm water, if over the sea."""
    warm = d["warm"].values
    if not warm.any():
        return pd.NaT
    k = np.flatnonzero(warm)[-1] + 1
    if k >= len(d) or np.isnan(d["sst_c"].values[k]):
        return pd.NaT
    return d["t"].values[k]


exits = pd.to_datetime(rec.groupby("sid").apply(exit_time, include_groups=False)).dropna()
comp = rec[rec["sid"].isin(exits.index)].copy()
comp["hours"] = (comp["t"] - comp["sid"].map(exits)).dt.total_seconds() / 3600
comp = comp[comp["hours"].between(-120, 48)]
comp["abs_lat"] = comp["lat"].abs()
PANELS = {"vmax_kt": "max wind (kt)", "sst_c": "SST under the storm (°C)", "abs_lat": "latitude (° from equator)",
          "speed_kmh": "forward speed (km/h)", "pole_km": "poleward motion (km/24 h)", "west_km": "westward motion (km/24 h)"}
fig, axes = plt.subplots(2, 3, figsize=(14, 6.5), sharex=True, layout="constrained")
for ax, col in zip(axes.flat, PANELS):
    q = comp.groupby("hours")[col].quantile([0.25, 0.5, 0.75]).unstack()
    ax.fill_between(q.index, q[0.25], q[0.75], color="#c46a3c", alpha=0.2)
    ax.plot(q.index, q[0.5], color="#c46a3c")
    ax.axvline(0, color="gray", linewidth=0.8)
    if col == "sst_c":
        ax.axhline(WARM_C, color="black", linestyle="--", linewidth=0.8)
    if col in ("pole_km", "west_km"):
        ax.axhline(0, color="gray", linewidth=0.8)
    ax.set_title(PANELS[col], fontsize=10)
for ax in axes[1]:
    ax.set_xlabel("hours from leaving warm water")
fig.suptitle(f"{len(exits)} storms leaving warm water: they peak before, then turn poleward and east, speed up and fade")
source_note(fig, CREDIT)

peak_t = rec.loc[rec.groupby("sid")["vmax_kt"].idxmax()].set_index("sid")["t"]
lead = ((exits - peak_t.reindex(exits.index)).dt.total_seconds() / 3600)
display(Markdown(f"- Storms that leave warm water over the sea: **{len(exits)}** of {len(storms)} ({len(exits) / len(storms):.0%}); the rest end over land or over warm water.\n"
                 f"- Their peak comes a median **{lead.median():.0f} hours** before they leave warm water (middle half {lead.quantile(0.25):.0f}–{lead.quantile(0.75):.0f} h)."))

### 7.1 Direction and speed against the water

Every record over water, by which way the storm moves over the next 24 hours and how fast, against the SST under it and its next 24-hour wind change.

In [ ]:
d = over_water.dropna(subset=["speed_kmh", "pole_km", "west_km"]).copy()
heading = np.degrees(np.arctan2(d["pole_km"], d["west_km"]))  # 0 = west, 90 = poleward, 180 = east
d["heading"] = pd.cut(heading, [-180, -135, -45, 45, 135, 180], labels=["east", "equatorward", "west", "poleward", "east "])
d["heading"] = d["heading"].astype(str).str.strip()
d["speed"] = pd.cut(d["speed_kmh"], [0, 10, 20, 30, 500], labels=["< 10", "10–20", "20–30", "≥ 30"])
by_heading = d.groupby("heading").agg(records=("dv24_kt", "size"), sst_c=("sst_c", "median"),
                                       sst_change_c=("sst_next24_c", lambda s: (s - d.loc[s.index, "sst_c"]).median()),
                                       dv24_kt=("dv24_kt", "mean")).loc[["west", "poleward", "east", "equatorward"]]
display(Markdown("**By heading** (median SST, median SST change over the 24 hours, mean wind change)"))
display(by_heading.round(2))

heat = d.assign(sst=pd.cut(d["sst_c"], [0, 26, 27, 28, 29, 40], labels=["< 26", "26–27", "27–28", "28–29", "≥ 29"]))
heat = heat.pivot_table(index="speed", columns="sst", values="dv24_kt", aggfunc="mean", observed=True)
fig, ax = plt.subplots(figsize=(7, 3.6), layout="constrained")
im = ax.imshow(heat.values, cmap="RdBu_r", vmin=-15, vmax=15, aspect="auto")
ax.set_xticks(range(heat.shape[1]), heat.columns)
ax.set_yticks(range(heat.shape[0]), heat.index)
for (i, j), v in np.ndenumerate(heat.values):
    ax.annotate(f"{v:+.0f}", (j, i), ha="center", va="center", fontsize=9)
ax.set(xlabel="SST under the storm (°C)", ylabel="forward speed (km/h)", title="Mean wind change over 24 h (kt): the water decides more than the speed")
fig.colorbar(im, ax=ax, label="kt / 24 h")
source_note(fig, CREDIT)

## 8. Hot seasons

Ours, beyond what 5097 claims. For each basin and season: the mean SST of its main development region in its peak months, against the season's storm count and energy. Southern Hemisphere seasons run from July to June and are named by the year they end in.

| Basin | Region | Peak months |
| --- | --- | --- |
| Atlantic | 10–20° N, 85–20° W | August–October |
| West Pacific | 5–20° N, 120–170° E | July–October |
| South Indian | 5–20° S, 40–135° E | January–March |
| South Pacific | 5–20° S, 135° E–150° W | January–March |

In [ ]:
REGION = {"Atlantic": ((10, 20), (275, 340), (8, 9, 10)), "West Pacific": ((5, 20), (120, 170), (7, 8, 9, 10)),
          "South Indian": ((-20, -5), (40, 135), (1, 2, 3)), "South Pacific": ((-20, -5), (135, 210), (1, 2, 3))}


def region_mean(sst, lat_range, lon_range):
    rows = (GRID_LAT >= lat_range[0]) & (GRID_LAT <= lat_range[1])
    cols = ((GRID_LON % 360) >= lon_range[0]) & ((GRID_LON % 360) <= lon_range[1])
    x = sst[rows][:, cols]
    w = np.broadcast_to(np.cos(np.radians(GRID_LAT[rows]))[:, None], x.shape)
    ok = np.isfinite(x)
    return float((x[ok] * w[ok]).sum() / w[ok].sum())


monthly = []
for m in MONTHS:
    sst = read_mur(m)
    for b, (la, lo, months) in REGION.items():
        if m.month in months:
            monthly.append((b, m.year, region_mean(sst, la, lo)))
season = pd.DataFrame(monthly, columns=["basin", "season", "region_sst_c"]).groupby(["basin", "season"]).mean().reset_index()
every = rec_all.assign(ace=np.where(rec_all["vmax_kt"] >= 35, rec_all["vmax_kt"] ** 2 / 1e4, 0.0)).groupby("sid").agg(
    basin=("basin", "first"), start=("t", "min"), ace=("ace", "sum"), lmi_kt=("vmax_kt", "max"))
every["season"] = np.where(every["basin"].str.startswith("South") & (every["start"].dt.month >= 7),
                           every["start"].dt.year + 1, every["start"].dt.year)
activity = every.groupby(["basin", "season"]).agg(storms=("ace", "size"), ace=("ace", "sum"),
                                                    majors=("lmi_kt", lambda v: (v >= 96).sum())).reset_index()
season = season.merge(activity, on=["basin", "season"], how="left").fillna({"storms": 0, "ace": 0, "majors": 0})
season = season[~(season["basin"].str.startswith("South") & (season["season"] == YEARS[0]))]  # its first half is before 2003

fig, axes = plt.subplots(1, 4, figsize=(14, 3.6), layout="constrained")
rows = []
for ax, b in zip(axes, BASINS):
    x = season[season["basin"] == b]
    ax.scatter(x["region_sst_c"], x["ace"], color=BASIN_COLOR[b], s=14)
    for _, r in x[x["season"].isin([2023, 2024])].iterrows():
        ax.annotate(str(int(r["season"])), (r["region_sst_c"], r["ace"]), fontsize=8)
    ax.set(title=b, xlabel="region SST in peak months (°C)")
    rows.append({"basin": b, "seasons": len(x),
                 "ρ SST–storms": spearmanr(x["region_sst_c"], x["storms"]).statistic,
                 "ρ SST–ACE": spearmanr(x["region_sst_c"], x["ace"]).statistic,
                 "ρ SST–major storms": spearmanr(x["region_sst_c"], x["majors"]).statistic,
                 "2023 SST rank": int(x["region_sst_c"].rank(ascending=False)[x["season"] == 2023].iloc[0]),
                 "2023 ACE rank": int(x["ace"].rank(ascending=False)[x["season"] == 2023].iloc[0])})
axes[0].set_ylabel("season ACE (10⁴ kt²)")
fig.suptitle("Season energy against the water in the main development region")
source_note(fig, CREDIT)
pd.DataFrame(rows).set_index("basin").round(2)

### 8.1 The warm pool breathes with the season

The area of each basin's water at or above 26.5 °C, month by month, against the number of storms active on an average day of that month. The eight storm years folded into one, as one season.

In [ ]:
BASIN_BOX = {"Atlantic": ((0, 45), (260, 360)), "West Pacific": ((0, 45), (100, 180)),
             "South Indian": ((-45, 0), (30, 135)), "South Pacific": ((-45, 0), (135, 240))}
cell_km2 = (111.2 * STEP) ** 2 * np.cos(np.radians(GRID_LAT))[:, None]


def warm_area(sst, lat_range, lon_range):
    rows = (GRID_LAT >= lat_range[0]) & (GRID_LAT <= lat_range[1])
    cols = ((GRID_LON % 360) >= lon_range[0]) & ((GRID_LON % 360) < lon_range[1])
    return float(np.where(sst[rows][:, cols] >= WARM_C, cell_km2[rows], 0).sum() / 1e6)


warm = []
for m in MONTHS[MONTHS.year.isin(STORM_YEARS)]:
    sst = read_mur(m)
    warm += [(b, m, warm_area(sst, *BASIN_BOX[b])) for b in BASINS]
warm = pd.DataFrame(warm, columns=["basin", "month", "warm_mkm2"])
days = rec.assign(day=rec["t"].dt.floor("D")).groupby(["basin", "day"])["sid"].nunique().rename("active").reset_index()
days["month"] = days["day"].dt.to_period("M")
ndays = pd.Series({m: m.days_in_month for m in MONTHS})  # active storms per day of the month
active = days.groupby(["basin", "month"])["active"].sum().reset_index()
active["active"] /= active["month"].map(ndays)
warm = warm.merge(active, on=["basin", "month"], how="left").fillna({"active": 0})
warm["moy"] = warm["month"].dt.month

fig, axes = plt.subplots(1, 4, figsize=(14, 3.6), layout="constrained")
rows = []
for ax, b in zip(axes, BASINS):
    clim = warm[warm["basin"] == b].groupby("moy")[["warm_mkm2", "active"]].mean()
    ax.bar(clim.index, clim["warm_mkm2"], color=BASIN_COLOR[b], alpha=0.35)
    ax2 = ax.twinx()
    ax2.plot(clim.index, clim["active"], color="black", marker=".")
    ax.set(title=b, xticks=range(1, 13), xticklabels=list("JFMAMJJASOND"))
    ax2.set_ylim(0, None)
    w = warm[warm["basin"] == b]
    rows.append({"basin": b, "ρ warm area – active storms (months)": spearmanr(w["warm_mkm2"], w["active"]).statistic,
                 "warm area, min–max month (million km²)": f"{clim['warm_mkm2'].min():.0f}–{clim['warm_mkm2'].max():.0f}"})
axes[0].set_ylabel("water ≥ 26.5 °C (million km², bars)")
ax2.set_ylabel("storms active per day (line)")
fig.suptitle("Storms come when the warm pool is largest")
source_note(fig, CREDIT)
pd.DataFrame(rows).set_index("basin").round(2)

## 9. The album's seasons as a score

The four seasons the album plays, drawn the way the music could read them: each storm's wind over time, coloured by the water under it, over the basin's warm area. A cross marks the moment a storm leaves warm water.

The South Pacific season of the earlier design (1997–98) is before MUR begins. In its place, the South Pacific season with the most named storms among the seasons this notebook has maps for.

In [ ]:
from matplotlib.collections import LineCollection

sp_seasons = every[(every["basin"] == "South Pacific") & every["season"].isin([2004, 2005, 2006, 2022, 2023, 2024])]
sp_best = int(sp_seasons.groupby("season").size().idxmax())
ALBUM = {  # basin, first day, last day
    "Atlantic 2023": ("Atlantic", "2023-06-01", "2023-11-30"),
    "West Pacific 2023": ("West Pacific", "2023-05-01", "2023-11-30"),
    "South Indian 2022–23": ("South Indian", "2022-12-01", "2023-04-30"),
    f"South Pacific {sp_best - 1}–{str(sp_best)[2:]}": ("South Pacific", f"{sp_best - 1}-11-01", f"{sp_best}-04-30"),
}
print("South Pacific named storms per season:", sp_seasons.groupby("season").size().to_dict())

fig, axes = plt.subplots(4, 1, figsize=(14, 13), layout="constrained")
score = []
for ax, (title, (b, t0, t1)) in zip(axes, ALBUM.items()):
    d = rec[(rec["basin"] == b) & rec["t"].between(t0, t1)]
    area = warm[(warm["basin"] == b) & warm["month"].between(pd.Period(t0, "M"), pd.Period(t1, "M"))]
    ax2 = ax.twinx()
    ax2.bar(area["month"].dt.to_timestamp() + pd.Timedelta(days=14), area["warm_mkm2"], width=27, color="#f2c6a8", alpha=0.5, zorder=0)
    ax2.set_ylabel("water ≥ 26.5 °C (million km²)", color="#b07050")
    ax.set_zorder(ax2.get_zorder() + 1)
    ax.patch.set_visible(False)
    for sid, s in d.groupby("sid"):
        x = matplotlib.dates.date2num(s["t"])
        pts = np.column_stack([x, s["vmax_kt"]]).reshape(-1, 1, 2)
        lc = LineCollection(np.concatenate([pts[:-1], pts[1:]], axis=1), cmap="RdYlBu_r", norm=plt.Normalize(24, 30), linewidth=2)
        lc.set_array(s["sst_c"].values[:-1])
        ax.add_collection(lc)
        if sid in exits.index and t0 <= str(exits[sid]) <= t1:
            ax.scatter(exits[sid], s.loc[s["t"] == exits[sid], "vmax_kt"], marker="x", color="black", zorder=4)
    ax.set(xlim=(pd.Timestamp(t0), pd.Timestamp(t1)), ylim=(0, 170), ylabel="max wind (kt)", title=title)
    days = pd.date_range(t0, t1, freq="D")
    active = d.assign(day=d["t"].dt.floor("D")).groupby("day")["sid"].nunique().reindex(days, fill_value=0)
    power = d.assign(p=d["vmax_kt"] ** 2).groupby("t")["p"].sum()
    move = d.dropna(subset=["pole_km", "west_km"])
    head = np.degrees(np.arctan2(move["pole_km"], move["west_km"]))
    score.append({"season": title, "named storms": d["sid"].nunique(), "most at once": int(active.max()),
                  "days without a storm": f"{(active == 0).mean():.0%}",
                  "leave warm water": int(sum(t0 <= str(exits.get(s_, pd.NaT)) <= t1 for s_ in d["sid"].unique() if s_ in exits.index)),
                  "time over warm water": f"{d['warm'][d['sst_c'].notna()].mean():.0%}",
                  "SST under storms, 5–95 % (°C)": f"{d['sst_c'].quantile(0.05):.1f}–{d['sst_c'].quantile(0.95):.1f}",
                  "wind, max (kt)": int(d["vmax_kt"].max()),
                  "moving west / poleward / east": "{:.0%} / {:.0%} / {:.0%}".format((head.abs() <= 45).mean(), head.between(45, 135).mean(), (head.abs() >= 135).mean()),
                  "peak wind power Σv² (10⁴ kt²)": round(power.max() / 1e4, 1)})
sm = plt.cm.ScalarMappable(cmap="RdYlBu_r", norm=plt.Normalize(24, 30))
fig.colorbar(sm, ax=axes, shrink=0.4, label="SST under the storm (°C)")
fig.suptitle("Each storm's wind, coloured by the water under it; × = leaves warm water")
source_note(fig, CREDIT)
pd.DataFrame(score).set_index("season").T

## 10. Message summary

| # | EIC page | Message | Key number (this notebook) | Track data |
| --- | --- | --- | --- | --- |
| 1 | [5097](https://svs.gsfc.nasa.gov/5097/) | Storms form over water warmer than ~26.5 °C | 87 % (Atlantic) to 100 % (West Pacific, South Pacific) of named storms are born over water at or above 26.5 °C; median 28.3–29.4 °C | SST under each storm record |
| 2 | [5097](https://svs.gsfc.nasa.gov/5097/) | Warmer water, more intense storms | Storms strengthen over water warmer than 27–28 °C and weaken below it; the share strengthening in 24 h rises from about 20 % under 26.5 °C to 77 % at 30 °C. The link is immediate. | 24-hour wind change against SST |
| 2a | ours | Time over warm water is the fuel | Hours over water ≥ 26.5 °C: ρ 0.83 with energy (ACE), 0.74 with peak wind; the mean SST alone: ρ ≈ 0 | Warm hours per storm |
| 3 | ours | High-energy storms | Top 10 %: 10 days as a named storm, 5,000 km, 94 % rapidly intensify, born at 11°; bottom half: 2 days, 1,600 km, 8 %, 15°. Same forward speed (17–19 km/h) | Storm table |
| 4 | ours | A storm's life | Peak about 36 h before leaving warm water; in those hours it turns poleward and east and its forward speed nearly doubles | Composite around leaving warm water |
| 5 | [40487](https://svs.gsfc.nasa.gov/gallery/nasaand-hurricanes/), ours | Warm pool and storm season | Warm area and active storms rise and fall together (ρ 0.58–0.75 by month) | Monthly warm area per basin |

## 11. From the data to music (ours)

The earlier designs each heard one side of a storm:

- **The season as voices**: every storm on the globe is a gliding voice; latitude is pitch (north higher), east–west travel from its birthplace is left–right, the ocean is a fixed bed.
- **The single storm as rhythm**: wind speed is tempo, the storm's spin pans the percussion around the listener, the 24-hour wind change shapes the melody.

What 5097 adds is the reason: the water under the storm. The data above says how the water can enter the sound without breaking either design.

### 11.1 What the data offers

| Finding | Data | What it suggests in sound | Why it fits |
| --- | --- | --- | --- |
| A line in the water | Weakening below ~27–28 °C, strengthening above (5.1) | A **tuning line**: a voice over water above the neutral SST is in tune and full; below it the voice goes flat and thin | One threshold, the one EIC draws, heard as in tune / out of tune; no listener needs a scale of degrees |
| The wind answers at once | Correlation highest for the window around the present (5.2) | The voice's colour changes on the step the storm crosses into cooler or warmer water, never before or after | Q2: what the screen shows (track colour, the white 26.5 °C line) and what is heard change together |
| Fuel adds up | Warm hours ρ 0.83 with ACE; top storms 204 h over warm water, bottom half 42 h (5.3, 6) | A voice **gathers weight** for every hour it spends over warm water: an octave below, then a fifth, then the rhythmic layer of the single-storm design | The longest-fed storms become the loudest, thickest parts; a short-lived storm stays a thin line |
| Rapid intensification | 94 % of the top storms against 8 % of the bottom half (6) | The single-storm melody shape: a leap upward when the wind rises 30 kt in 24 h | Already designed and heard in the earlier album; the water explains when it happens |
| The arc of a life | Over warm water, moving west; ~2 days before leaving it: turning poleward and east, speeding up; peak ~36 h before; then fading (7) | A **phrase**: the voice rises (poleward = higher pitch), swings back from left to right (east), then **cadences** as it crosses the line | The existing pitch and pan mappings already play the turn; the crossing is the resolution |
| Leaving warm water | 1–6 storms per album season (9) | A rare **event**: the voice's last note falls to the tonic and decays; on screen the 26.5 °C line flashes where it is crossed | Rare enough to stay an event, not a pattern |
| Forward speed | The same for strong and weak storms (6); doubles only near the end (7) | Not a main channel. At most the glide time between notes | Spending a channel on it would suggest a link the data does not show |
| The warm pool | Warm area 10–25 million km² over the year, follows active storms (8.1) | The ocean bed is no longer fixed: its swell and depth follow the basin's warm area | The bed becomes the reason storms can exist at all |

### 11.2 A combined design

Three layers, each tied to one side of the data:

1. **The ocean (bed)**: low drone and pad, as in the season design, but swelling with the basin's warm area month by month. Heat is heard as pressure (a wider, lower, slowly beating chord), not as brightness: warm water is bad news here and must not sound cheerful (album guide).
2. **The storms (voices)**: each storm is a gliding voice as before (latitude → pitch, east–west → pan). Two additions from the water:
   - **Tuning to the water**: over water above the basin's neutral SST, the voice sits in the key with its full harmonics. Below it, the voice bends flat (up to a quarter tone at 2 °C below), its filter closes and its vibrato slows: the engine running out of fuel.
   - **Weight from fuel**: every 48 hours over warm water adds one layer (sub-octave, fifth, then a rhythmic pulse at the storm's own rate from the single-storm design: the stronger the wind, the finer the subdivision on the shared grid). A storm that keeps feeding becomes a chord; one that starves stays a single line.
3. **The crossing (event)**: when a storm leaves warm water, its voice resolves to the tonic and fades over a bar, and the 26.5 °C line on the globe lights where it crossed.

What a listener should hear: storms appear over the swelling sea, rise and thicken while they stay over warm water, swing east and climb as they turn, then fall out of tune and resolve the moment they leave it. In 2023 the sea swells the most.

### 11.3 Open choices

- **Tempo**: the single-storm album ties tempo to wind. With up to five storms at once a season cannot have one storm's tempo. The design above keeps one tempo and gives each storm its own subdivision instead. The alternative is a global tempo from the season's total wind power (Σv², up to 2.7 × 10⁴ kt² in these seasons), which would make the busiest weeks rush.
- **Neutral SST per basin or one line**: the data puts it at 26.9 °C (Atlantic) to 28.1 °C (West Pacific, South Pacific). EIC's 26.5 °C is the line drawn on screen; the tuning can use each basin's own value or the same 26.5 °C for all.
- **South Pacific season**: 1997–98 is before MUR. Within the years mapped here, the best season has 8–9 named storms and is quiet for 75 % of its days.